In [ ]:
# -*- coding: utf-8 -*-
from abc import ABC, abstractmethod
from typing import Dict

import torch as th
from torch import nn

from torch import nn
from typing import Callable
import torch as th
from torch.nn.init import xavier_normal_, normal_
from torch.nn import functional as F


In [ ]:
def hermite(x: th.Tensor, n: int) -> th.Tensor:
    h_s = [th.ones(*x.size(), device=x.device), x]

    for i in range(1, n):
        h_s.append(x * h_s[i] - i * h_s[i - 1])

    return th.slice_copy(th.stack(h_s, dim=-1), -1, 1) / th.exp(
        th.lgamma(th.arange(2, n + 2, device=x.device)) / 2
    )


class Hermite(nn.Module):
    def __init__(self, n: int) -> None:
        super().__init__()
        self.__n = n

    def forward(self, x: th.Tensor) -> th.Tensor:
        return hermite(x, self.__n)

    def get_size(self) -> int:
        return self.__n

In [ ]:
batch_size = 3
in_channels = 2
out_channels = 4
size = 8

kernel_size = 4
stride = 2
padding = 1

n = 5

output_size = stride * (size - 1) + kernel_size - 2 * padding

In [ ]:
output_size

In [ ]:
x = th.randn(batch_size, in_channels, size, size)

In [ ]:
c = th.randn(in_channels, out_channels, kernel_size * kernel_size, 1, n)

In [ ]:
out = th.sum(x.view(batch_size, in_channels, 1, 1, size * size, 1) * c, dim=-1)
out = th.sum(out, dim=1)
out = out.view(batch_size, out_channels * kernel_size**2, -1)
out = F.fold(out, output_size, kernel_size, dilation=1, padding=padding, stride=stride)

In [ ]:
out.size()